In [ ]:
import os, time, copy, random
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models

# Reproducibility
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

In [ ]:
from google.colab import files
files.upload()

In [ ]:
import zipfile

# Unzip (adjust filename if needed)
zip_path = "train_data.zip"
assert os.path.exists(zip_path), f"Zip not found: {zip_path}"

with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(".")

print("Files here:", os.listdir("."))
print("train_data sample:", os.listdir("train_data")[:10])

In [ ]:
import os
from torchvision import datasets

DATA_DIR = "./train_data"
assert os.path.exists(DATA_DIR), f"Not found: {DATA_DIR}"

ds_check = datasets.ImageFolder(DATA_DIR)
print("classes:", ds_check.classes)
print("class_to_idx:", ds_check.class_to_idx)

expected = ['Convertible', 'Coupe', 'Pickup', 'SUV', 'Sedan']
assert ds_check.classes == expected, f"Mismatch! Expected {expected}, got {ds_check.classes}"
print("Class order is correct!")

In [ ]:
import copy, numpy as np
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

DATA_DIR = "./train_data"
full_ds = datasets.ImageFolder(DATA_DIR)

expected = ['Convertible', 'Coupe', 'Pickup', 'SUV', 'Sedan']
assert full_ds.classes == expected, f"Expected {expected}, got {full_ds.classes}"

# TRAIN: closer to Gradescope pipeline but still with augmentation
train_tfms = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandAugment(num_ops=2, magnitude=7),
    transforms.ToTensor(),
    transforms.RandomErasing(p=0.2, scale=(0.02, 0.10), ratio=(0.3, 3.3)),
])


val_tfms = transforms.Compose([
    transforms.Resize(224),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
])

targets = np.array(full_ds.targets)
idxs = np.arange(len(full_ds))
train_idx, val_idx = [], []

# stratified 85/15 split per class
for c in range(5):
    c_idxs = idxs[targets == c]
    np.random.shuffle(c_idxs)
    split = int(0.85 * len(c_idxs))
    train_idx.extend(c_idxs[:split])
    val_idx.extend(c_idxs[split:])

train_idx = np.array(train_idx)
val_idx   = np.array(val_idx)

train_base = copy.deepcopy(full_ds); train_base.transform = train_tfms
val_base   = copy.deepcopy(full_ds); val_base.transform   = val_tfms

train_ds = Subset(train_base, train_idx)
val_ds   = Subset(val_base, val_idx)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds, batch_size=64, shuffle=False, num_workers=2, pin_memory=True)

print("Train size:", len(train_ds), "Val size:", len(val_ds))

In [ ]:
import torch
import torch.nn as nn
from torchvision import models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class NormalizeInModel(nn.Module):
    def __init__(self, mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)):
        super().__init__()
        self.register_buffer("mean", torch.tensor(mean).view(1,3,1,1))
        self.register_buffer("std",  torch.tensor(std).view(1,3,1,1))
    def forward(self, x):
        return (x - self.mean) / self.std

class CarNet(nn.Module):
    def __init__(self, num_classes=5, use_flip_tta=True):
        super().__init__()
        self.norm = NormalizeInModel()
        self.use_flip_tta = use_flip_tta

        backbone = models.efficientnet_b1(weights=models.EfficientNet_B1_Weights.DEFAULT)
        in_f = backbone.classifier[1].in_features
        backbone.classifier[1] = nn.Linear(in_f, num_classes)
        self.backbone = backbone

    def forward_once(self, x):
        x = self.norm(x)
        return self.backbone(x)

    def forward(self, x):
        if not self.use_flip_tta:
            return self.forward_once(x)
        # deterministic TTA inside the model (traceable)
        logits1 = self.forward_once(x)
        logits2 = self.forward_once(torch.flip(x, dims=[3]))  # horizontal flip
        return (logits1 + logits2) * 0.5

model = CarNet(use_flip_tta=True).to(device)

In [ ]:
import copy
import torch.nn.functional as F

def mixup_data(x, y, alpha=0.2):
    if alpha <= 0:
        return x, y, y, 1.0
    lam = np.random.beta(alpha, alpha)
    perm = torch.randperm(x.size(0), device=x.device)
    x_mix = lam * x + (1 - lam) * x[perm]
    y_a, y_b = y, y[perm]
    return x_mix, y_a, y_b, lam

@torch.no_grad()
def update_ema(ema_model, model, decay=0.999):
    msd = model.state_dict()
    for k, v in ema_model.state_dict().items():
        if v.dtype.is_floating_point:
            v.mul_(decay).add_(msd[k].detach(), alpha=1 - decay)
        else:
            v.copy_(msd[k])

@torch.no_grad()
def eval_acc(m, loader):
    m.eval()
    correct = total = 0
    for x,y in loader:
        x,y = x.to(device), y.to(device)
        pred = m(x).argmax(1)
        correct += (pred == y).sum().item()
        total += y.size(0)
    return correct / total

In [ ]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss(label_smoothing=0.08)

# EMA copy
ema_model = copy.deepcopy(model).to(device)
for p in ema_model.parameters():
    p.requires_grad = False

best_wts = None
best_val = 0.0

# -------- Stage 1: train head only --------
for p in model.parameters():
    p.requires_grad = False
for p in model.backbone.classifier.parameters():
    p.requires_grad = True

opt = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=3e-3, weight_decay=1e-4)
sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=5)

for epoch in range(5):
    model.train()
    for x,y in train_loader:
        x,y = x.to(device), y.to(device)
        opt.zero_grad(set_to_none=True)

        x_mix, y_a, y_b, lam = mixup_data(x, y, alpha=0.2)
        out = model(x_mix)
        loss = lam * criterion(out, y_a) + (1-lam) * criterion(out, y_b)

        loss.backward()
        opt.step()
        update_ema(ema_model, model, decay=0.999)

    sched.step()
    val_acc = eval_acc(ema_model, val_loader)  # evaluate EMA
    if val_acc > best_val:
        best_val = val_acc
        best_wts = copy.deepcopy(ema_model.state_dict())
    print(f"[Head] {epoch+1}/5  EMA val_acc={val_acc:.4f}  best={best_val:.4f}")

# -------- Stage 2: fine-tune all --------
# load best EMA weights into model before fine-tuning
model.load_state_dict(best_wts)
ema_model.load_state_dict(best_wts)

for p in model.parameters():
    p.requires_grad = True

opt = optim.AdamW(model.parameters(), lr=5e-4, weight_decay=2e-4)
sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=35)

for epoch in range(35):
    model.train()
    for x,y in train_loader:
        x,y = x.to(device), y.to(device)
        opt.zero_grad(set_to_none=True)

        x_mix, y_a, y_b, lam = mixup_data(x, y, alpha=0.2)
        out = model(x_mix)
        loss = lam * criterion(out, y_a) + (1-lam) * criterion(out, y_b)

        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        update_ema(ema_model, model, decay=0.999)

    sched.step()
    val_acc = eval_acc(ema_model, val_loader)
    if val_acc > best_val:
        best_val = val_acc
        best_wts = copy.deepcopy(ema_model.state_dict())
    print(f"[FT] {epoch+1}/35  EMA val_acc={val_acc:.4f}  best={best_val:.4f}")

# Final best model = EMA weights
model.load_state_dict(best_wts)
model.eval()
print("Final best EMA val_acc:", best_val)

In [ ]:
import os, torch

model_cpu = model.to("cpu").eval()
dummy_input = torch.randn(1, 3, 224, 224)
traced = torch.jit.trace(model_cpu, dummy_input)
traced.save("model.pt")

print("Saved model.pt")
print("File size (MB):", os.path.getsize("model.pt")/(1024*1024))

In [ ]:
from google.colab import files
files.download("model.pt")